##**NAME: AFRA KHURRAM ANSARI**
##**ROLL NO: CS-23008**
##**ML ASSIGNMENT**

# DataSet Selection

For this assignment, I selected the Titanic dataset, which belongs to the transportation and disaster domain.

**Source of Dataset:**
The dataset is taken from Kaggle (Titanic: Machine Learning from Disaster). It is based on real historical records of passengers who were traveling on the RMS Titanic.

**Brief Description of the Dataset:**

This dataset contains information about 891 passengers who were on the Titanic. Each row represents one passenger. There are 12 columns in the dataset that describe different details about the passengers, such as their age, gender, class, and whether they survived or not. The main goal of using this dataset is to study the factors that affected passenger survival during the Titanic accident.

In [1]:
# Necessary Imports
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

#Data Understanding

In [2]:
# Loading the dataset
data = pd.read_csv('Titanic-Dataset.csv')
data.head()

FileNotFoundError: [Errno 2] No such file or directory: 'Titanic-Dataset.csv'

In [ ]:
# Exploring the structure of data
data.info()

In [ ]:
data.describe()

In [ ]:
data.shape

**Structure of the Dataset**

*PassengerId*: A unique ID given to each passenger

*Survived*: Shows whether the passenger survived (0 = did not survive, 1 = survived)

*Pclass*: The class in which the passenger was traveling (1st, 2nd, or 3rd)

*Name*: Name of the passenger

*Sex*: Gender of the passenger

*Age*: Age of the passenger

*SibSp*: Number of siblings or spouses traveling with the passenger

*Parch*: Number of parents or children traveling with the passenger

*Ticket*: Ticket number

*Fare*: Fare paid by the passenger

*Cabin*: Cabin number

*Embarked*: Port from where the passenger boarded the ship
(C = Cherbourg, Q = Queenstown, S = Southampton)

In [ ]:
# Checking the count of null values
data.isnull().sum()

In [ ]:
# Checking duplicate rows
data.duplicated().sum()

# Data Cleaning

## Handling Missing Values

In [ ]:
# Checking how age is distributed using skew property
# If the value is close to 0, the data is approx symmetric.
# If the value is positive, the data is right-skewed (more young ages and fewer very high ages).
# If the value is negative, the data is left-skewed (more older ages and fewer very low ages).
data['Age'].skew()

In [ ]:
# Plotting to check distribution
sns.histplot(data['Age'], bins=30, kde=True)
plt.title("Age Distribution")
plt.show()

In [ ]:
# Age data is positively skewed, the mean is not a good choice, it can be influenced by outliers.
# So, we fill the missing Age values using the median Age of each Passenger Class and Gender group to get reasonable values.
data['Age'] = data.groupby(['Pclass','Sex'])['Age'].transform( lambda x: x.fillna(x.median()) )

In [ ]:
# Since, many values are missing in Cabin column, so we drop this column
data.drop(columns=['Cabin'], inplace=True)

In [ ]:
data['Embarked'].value_counts()

In [ ]:
# Embarked is a categorical column with few missing values.
# We replace the missing values with the most common port of embarkation
data['Embarked'].fillna(data['Embarked'].mode()[0], inplace=True)


In [ ]:
# Checking the null values after cleaning
data.isnull().sum()

## Checking for inconsistent values

In [ ]:
# Check max and min values in age to see if there is any unexpected or invalid age
print(f"  Min: {data['Age'].min()}")
print(f"  Max: {data['Age'].max()}")

In [ ]:
# Check unique values in Embarked to see if there is any unexpected or invalid gender inconsistency in upper or lowercase
data['Sex'].unique()

In [ ]:
# Check unique values in Embarked to see if there is any unexpected or invalid data
data['Embarked'].unique()

In [ ]:
# Check max and min values in fare to see if there is any unexpected or invalid fare
print(f"  Min: {data['Fare'].min()}")
print(f"  Max: {data['Fare'].max()}")

In [ ]:
# Count how many passengers paid zero fare
zero_fares = data[data['Fare'] == 0].shape[0]
print(f"  Zero fares: {zero_fares} ({zero_fares/len(data)*100:.2f}%)")
if zero_fares > 0:
    print("Some passengers have zero fare (crew members?)")

In [ ]:
# Check unique values in passenger class to see if any invalid or negative class is present
data['Pclass'].unique()

In [ ]:
# Check max and min values in siblings/spouses to see if there is any unexpected or invalid data
print(f"  Min: {data['SibSp'].min()}")
print(f"  Max: {data['SibSp'].max()}")

In [ ]:
# Check max and min values in parent/children to see if there is any unexpected or invalid data
print(f"  Min: {data['Parch'].min()}")
print(f"  Max: {data['Parch'].max()}")

In [ ]:
# Check unique values of target variable
data['Survived'].unique()

In [ ]:
# Check if children paid very high fare
data[(data['Age'] < 5) & (data['Fare'] > 200)]

In [ ]:
print("Checking Fare vs Pclass Consistency:")
for pclass in [1, 2, 3]:
    pclass_data = data[data['Pclass'] == pclass]
    print(f"Class {pclass}:")
    print(f" Fare range: ${pclass_data['Fare'].min():.2f} - ${pclass_data['Fare'].max():.2f}")
    print(f" Mean fare: ${pclass_data['Fare'].mean():.2f}")

low_fare_1st = data[(data['Pclass'] == 1) & (data['Fare'] < 20 )]
if len(low_fare_1st) > 0:
    print(f"\n {len(low_fare_1st)} 1st class passengers with fare < $20")
    print(low_fare_1st[['PassengerId', 'Name', 'Pclass', 'Fare']])

high_fare_3rd = data[(data['Pclass'] == 3) & (data['Fare'] > 50)]
if len(high_fare_3rd) > 0:
    print(f"\n {len(high_fare_3rd)} 3rd class passengers with fare > $50")
    print(high_fare_3rd[['PassengerId', 'Name', 'Pclass', 'Fare']])

# A few first-class passengers are found to have very low or zero fare values.
# Though this appears strange but such cases can be historically possible (for example, special tickets or fares paid by others).
# So, these records will be kept for analysis

# Feature Engineering

In [ ]:
# Count how many times each ticket appears to detect group or family travel
ticket_counts = data['Ticket'].value_counts()
data['TicketFrequency'] = data['Ticket'].map(ticket_counts)
print(f"   Ticket frequency range: {data['TicketFrequency'].min()} to {data['TicketFrequency'].max()}")

In [ ]:
# Create FamilySize
data['FamilySize'] = data['SibSp'] + data['Parch'] + 1

In [ ]:
# Extracting the title from the Name column
data['Title'] = data['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

In [ ]:
print(f"   Unique titles: {data['Title'].nunique()}")
print(f"   Title distribution:\n{data['Title'].value_counts()}")

In [ ]:
# Create a mapping dictionary to group rare titles into common categories
title_mapping = { 'Mr': 'Mr', 'Miss': 'Miss', 'Mrs': 'Mrs', 'Master': 'Master', 'Dr': 'Rare', 'Rev': 'Rare', 'Col': 'Rare', 'Major': 'Rare', 'Mlle': 'Miss', 'Countess': 'Rare', 'Ms': 'Miss', 'Lady': 'Rare', 'Jonkheer': 'Rare', 'Don': 'Rare', 'Dona': 'Rare', 'Mme': 'Mrs', 'Capt': 'Rare', 'Sir': 'Rare' }
data['Title'] = data['Title'].map(title_mapping)
print(f"\n   Grouped titles:\n{data['Title'].value_counts()}")

In [ ]:
# Check mismatch between Sex and Title
data[(data['Sex'] == 'male') & (data['Title'].isin(['Mrs','Miss']))]

In [ ]:
# Check Master with high age
data[(data['Title'] == 'Master') & (data['Age'] > 25)]

In [ ]:
# Check extreme family sizes
data[data['FamilySize'] > 10]

In [ ]:
print(f"Family size range: {data['FamilySize'].min()} to {data['FamilySize'].max()}")

In [ ]:
# Converting continuous Age values into age groups for easier analysis
data['AgeGroup'] = pd.cut(data['Age'], bins=[0, 12, 18, 35, 60, 100],
                               labels=['Child', 'Teenager', 'Adult', 'Middle-aged', 'Senior'])
print(f"   Age group distribution:\n{data['AgeGroup'].value_counts()}")

In [ ]:
# Converting continuous fare values into fare groups for easier analysis
data['FareBin'] = pd.qcut(data['Fare'], q=4, labels=['Low', 'Medium', 'High', 'Very High'], duplicates='drop')
print(f"Fare bin distribution:\n{data['FareBin'].value_counts()}")

In [ ]:
# Encoding the categorical values into numbers so they can be used by ML models
data['Sex'] = data['Sex'].map({'male': 0, 'female': 1})

In [ ]:
# Convert the categorical column 'Embarked' into dummy variables (one-hot encoding)
# drop_first=True is used to remove one category and avoid the dummy variable trap
data['Embarked_orig'] = data['Embarked']
data = pd.get_dummies(data, columns=['Embarked'], prefix='Embarked', drop_first=True)

In [ ]:
data['Title'].unique()

In [ ]:
# One-hot encode the Title column for use in ML models
data = pd.get_dummies(data, columns=['Title'], prefix='Title')

In [ ]:
data['AgeGroup'].unique()

In [ ]:
# One-hot encode the AgeGroup column for use in ML models
data['Age_orig'] = data['Age']
data = pd.get_dummies(data, columns=['AgeGroup'], prefix='AgeGroup')

In [ ]:
# One-hot encode the FareBin column for use in ML models
data['Fare_orig'] = data['Fare']
data = pd.get_dummies(data, columns=['FareBin'], prefix='FareBin')

In [ ]:
data['IsAlone'] = (data['FamilySize'] == 1).astype(int)
print(f"Passengers traveling alone: {data['IsAlone'].sum()}")

# Exploratory Data Analysis (EDA) and Visualization

In [ ]:
# Checking how many samples are there for survived and died people
survival_rate = data['Survived'].value_counts(normalize=True) * 100
survival_rate

In [ ]:
# bar chart showing the number of passengers who survived vs died
plt.figure(figsize=(5,3))
data['Survived'].value_counts().plot(kind='bar', color=['#ffb6c1', '#87cefa'])
plt.title('Survival Distribution')
plt.xlabel('Survived (0 = No, 1 = Yes)')
plt.ylabel('Count')
plt.xticks([0,1], ['Died', 'Survived'], rotation=0)
plt.show()

In [ ]:
# Percentage table
gender_survival = pd.crosstab(data['Sex'], data['Survived'], normalize='index') * 100
print(gender_survival)

In [ ]:
# bar chart showing the number of passengers who survived and died grouped by gender
plt.figure(figsize=(5,3))
sns.countplot(x='Survived', hue='Sex', data=data, palette=[ '#ffd3b6', '#ffaaa5'])
plt.title('Survival by Gender')
plt.xlabel('Survived')
plt.ylabel('Count')
plt.xticks([0,1], ['Died', 'Survived'])
plt.show()

In [ ]:
class_survival = pd.crosstab(data['Pclass'], data['Survived'], normalize='index') * 100
print(class_survival)

In [ ]:
# bar chart showing the number of passengers who survived and died grouped by passenger class
pd.crosstab(data['Pclass'], data['Survived']).plot(kind='bar', color=['#c8a2c8', '#b0e0e6'], figsize=(5,3))
plt.title('Survival by Passenger Class')
plt.xlabel('Passenger Class')
plt.ylabel('Count')
plt.xticks([0,1,2], ['1st Class', '2nd Class', '3rd Class'], rotation=0)
plt.legend(['Died', 'Survived'])
plt.show()

In [ ]:
# Plotting age distribution by survival and death
plt.figure(figsize=(5,3))
sns.histplot(data=data, x='Age', hue='Survived', bins=30, palette=['#d62728','#2ca02c'], alpha=0.5)
plt.title('Age Distribution by Survival')
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.show()

In [ ]:
embarked_survival = pd.crosstab(data['Embarked_orig'], data['Survived'], normalize='index') * 100
print(embarked_survival)

In [ ]:
# Showing proportion of passengers who survived from each boarding port
embarked_survival.plot(kind='bar', color=['#a8e6cf','#ffaaa5'], figsize=(5, 3))
plt.title('Survival by Embarked Port')
plt.xlabel('Embarked Port')
plt.ylabel('Percentage')
plt.legend(['Died', 'Survived'])
plt.show()

In [ ]:
family_survival = data.groupby('FamilySize')['Survived'].mean() * 100
print(family_survival)

In [ ]:
# Plotting survival rate by family size
family_survival.plot(kind='bar', color='#ffb347', figsize=(5, 3))
plt.title('Survival Rate by Family Size')
plt.xlabel('Family Size')
plt.ylabel('Survival Percentage')
plt.show()

In [ ]:
# Convert all boolean columns to integers
bool_cols = data.select_dtypes(include='bool').columns
data[bool_cols] = data[bool_cols].astype(int)

In [ ]:
data.columns

In [ ]:
numeric_features = [
    'Survived',
    'Pclass',
    'Sex',
    'Age_orig',
    'SibSp',
    'Parch',
    'Fare_orig',
    'TicketFrequency',
    'FamilySize',
    'IsAlone'
]
# Compute correlation matrix
corr = data[numeric_features].corr()

# Values close to +1: Strong positive correlation
# Values close to -1: Strong negative correlation
# Values close to 0: Little to no linear relationship

# Plotting the heatmap
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', center=0)
plt.title('Correlation of Numeric Features', fontsize=14)
plt.show()

In [ ]:
# Dropping columns that are not needed for modeling in future
data = data.drop(columns=['PassengerId', 'Name', 'Ticket', 'Age', 'Fare', 'Embarked_orig'])

In [ ]:
data.shape

## Summary of Analysis

This analysis to explore the Titanic dataset was to understand the factors that influenced passenger survival during the disaster. Through data cleaning, feature engineering, and exploratory data analysis, several important patterns got clear.

## Key Findings

- Females had a significantly higher survival rate ~74% compared to males ~19%
- This shows the "ladies and children first" evacuation protocol
- Gender shows one of the strongest correlations with survival in this analysis
- 1st class passengers had ~63% survival rate
- 2nd class passengers had ~47% survival rate
- 3rd class passengers had only ~24% survival rate
- This shows that socioeconomic status affected access to lifeboats
- Children (under 12) had higher survival rates than adults
- The age distribution shows that younger passengers were more likely to survive
- This shows the prioritization of children during evacuation

- Family Size Had a Non-Linear Effect
- Passengers traveling alone had lower survival rates (~30%)
- Small families (2-4 members) had the highest survival rates (~50-70%)
- Very large families (>6 members) had lower survival rates
- This shows that having some family support helped, but very large groups faced coordination challenges
- Passengers from Cherbourg had ~55% survival rate
- Passengers from Queenstown had ~39% survival rate
- Passengers from Southampton had ~34% survival rate
- This may be correlated with passenger class, as Cherbourg had more 1st class passengers


- Family size as a combined feature (SibSp + Parch + 1) provided better insights than individual features
- Age grouping helped identify that children and young adults had different survival patterns
- Fare binning showed clear correlation between ticket price and survival, showing class differences

 So, Multiple factors interacted to determine survival, not just a single variable

## Reflections and Learnings

- Handling missing data using domain-appropriate methods
- Feature engineering to create meaningful variables from existing data
- One-hot encoding for categorical variables
- Creating effective visualizations to communicate findings

